# 📘 智能体架构 3：ReAct (推理 + 行动)

欢迎来到我们系列的第三本笔记本。现在我们将探索 **ReAct**，这是一个连接简单工具使用与复杂多步问题解决之间的关键架构。ReAct 代表 **推理 + 行动**，其核心创新在于它使智能体能够动态地推理问题，根据推理采取行动，观察结果，然后再次推理。

这一模式将智能体从静态工具调用者转变为自适应问题解决者。为了突出其强大之处，我们将首先构建一个**基本的、单次工具使用智能体**并展示其在复杂任务上的局限性。然后，我们将构建一个完整的 ReAct 智能体，演示其迭代的 `思考 → 行动 → 观察` 循环如何使其在基本智能体失败的地方取得成功。

### 定义
**ReAct** 架构是一种设计模式，智能体在其中交错执行推理步骤和行动。智能体不是预先规划所有步骤，而是生成关于其直接下一步的想法，采取行动（如调用工具），观察结果，然后使用该新信息生成其下一个想法和行动。这创建了一个动态和自适应循环。

### 高层工作流程

1. **接收目标：** 智能体被赋予一个复杂任务。
2. **思考（推理）：** 智能体生成一个内部想法，例如：*"要回答这个问题，我首先需要找到信息 X。"*
3. **行动：** 基于其想法，智能体执行一个动作，通常是调用工具（例如，`search_api('X')`）。
4. **观察：** 智能体接收来自工具的结果。
5. **重复：** 智能体将观察结果整合到其上下文中并返回步骤 2，生成新的想法（例如，*"好的，现在有了 X，我需要用它来找到 Y。"*）。此循环持续直到满足总体目标。

### 适用场景 / 应用
* **多跳问题回答：** 当回答问题需要按顺序查找多条信息时（例如，"制作 iPhone 的公司 CEO 是谁？"）。
* **网络导航与研究：** 智能体可以搜索起始点，阅读结果，然后根据所学内容决定新的搜索查询。
* **交互式工作流程：** 任何环境动态且无法提前知道解决方案完整路径的任务。

### 优缺点
* **优点：**
    * **自适应和动态：** 可以根据新信息即时调整其计划。
    * **处理复杂性：** 擅长需要链接多个依赖步骤的问题。
* **缺点：**
    * **更高的延迟和成本：** 涉及多个顺序 LLM 调用，使其比单次方法更慢且更昂贵。
    * **循环风险：** 引导不当的智能体可能会陷入重复、无效果的想法和行动循环中。

## 阶段 0：基础与环境设置

我们将从我们的标准设置过程开始：安装库并为 OpenAI、LangSmith 和我们的 Tavily 网络搜索工具配置 API 密钥。

### 步骤 0.1：安装核心库

**我们要做什么：**
我们将安装本项目系列的标准库套件。

In [8]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv tavily-python

### 步骤 0.2：导入库和设置密钥

**我们要做什么：**
我们将导入必要的模块并从 `.env` 文件加载我们的 API 密钥。

**需要的操作：** 在此目录中创建一个包含密钥的 `.env` 文件：
```
OPENAI_API_KEY="your_openai_api_key_here"
OPENAI_API_BASE_URL="your_openai_api_base_url_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
TAVILY_API_KEY="your_tavily_api_key_here"
```

In [9]:
import os
from typing import Annotated
from dotenv import load_dotenv

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field

# LangGraph components
from langgraph.graph import StateGraph, END
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - ReAct (OpenAI)"

# Check that the keys are set
for key in ["OPENAI_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：基本方法 - 单次工具使用者

要理解为什么 ReAct 如此强大，我们必须首先看看没有它会发生什么。我们将构建一个"基本"智能体，它可以使用工具，但只能使用一次。它将分析用户的查询，进行一次工具调用，然后尝试基于那一条信息形成最终答案。

### 步骤 1.1：构建基本智能体

**我们要做什么：**
我们将定义与之前相同的工具和 LLM，但我们将它们连接到一个简单的线性图中。智能体只有一次调用工具的机会，然后工作流程结束。没有循环。

In [10]:
from typing import TypedDict

console = Console()

# Define the state for our graphs
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# Define the tool and LLM
model = os.environ.get("OPENAI_API_MODEL", "gpt-4o")
base_url = os.environ.get("OPENAI_API_BASE_URL", "https://api.openai.com/v1")
search_tool = TavilySearchResults(max_results=2, name="web_search")
llm = ChatOpenAI(model=model, base_url=base_url, temperature=0)
llm_with_tools = llm.bind_tools([search_tool])

# Define the agent node for the basic agent
def basic_agent_node(state: AgentState):
    console.print("--- BASIC AGENT: Thinking... ---")
    # Note: We provide a system prompt to encourage it to answer directly after one tool call
    system_prompt = "You are a helpful assistant. You have access to a web search tool. Answer the user's question based on the tool's results. You must provide a final answer after one tool call."
    messages = [("system", system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Define the basic, linear graph
basic_graph_builder = StateGraph(AgentState)
basic_graph_builder.add_node("agent", basic_agent_node)
basic_graph_builder.add_node("tools", ToolNode([search_tool]))

basic_graph_builder.set_entry_point("agent")
# After the agent, it can only go to tools, and after tools, it MUST end.
basic_graph_builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": "__end__"})
basic_graph_builder.add_edge("tools", END)

basic_tool_agent_app = basic_graph_builder.compile()

print("Basic single-shot tool-using agent compiled successfully.")

Basic single-shot tool-using agent compiled successfully.


### 步骤 1.2：在多步问题上测试基本智能体

**我们要做什么：**
现在我们将给基本智能体一个需要多个依赖步骤才能解决的问题。这将暴露其根本弱点。

In [11]:
multi_step_query = "Who is the current CEO of the company that created the sci-fi movie 'Dune', and what was the budget for that company's most recent film?"

console.print(f"[bold yellow]Testing BASIC agent on a multi-step query:[/bold yellow] '{multi_step_query}'\n")

basic_agent_output = basic_tool_agent_app.invoke({"messages": [("user", multi_step_query)]})

console.print("\n--- [bold red]Final Output from Basic Agent[/bold red] ---")
console.print(Markdown(basic_agent_output['messages'][-1].content))

Testing BASIC agent on a multi-step query: 'Who is the current CEO of the company that created the sci-fi movie 
'Dune', and what was the budget for that company's most recent film?'

--- BASIC AGENT: Thinking... ---

--- Final Output from Basic Agent ---

[{"title": "Dune (2021 film) - Wikipedia", "url": "https://en.wikipedia.org/wiki/Dune_(2021_film)", "content": "|  
Release poster |\n| Directed by | Denis Villeneuve |\n| Screenplay by |  Jon Spaihts  Denis Villeneuve  Eric Roth  
|\n| Based on | Dune "Dune (novel)") by Frank Herbert |\n| Produced by |  Mary Parent  Denis Villeneuve  Cale      
Boyter  Joe Caracciolo Jr. |\n| Starring |  Timothée Chalamet  Rebecca Ferguson  Oscar Isaac  Josh Brolin  Stellan 
Skarsgård  Dave Bautista  Stephen M. Henderson  Zendaya  Chang Chen  Sharon Duncan-Brewster  Charlotte Rampling    
Jason Momoa  Javier Bardem |\n| Cinematography | Greig Fraser |\n| Edited by | Joe Walker "Joe Walker (film        
editor)") |\n| Music by | Hans Zimmer |\n| Production company | Legendary Pictures |\n| Distributed by | Warner    
Bros. Pictures |\n| Release dates |  September 3, 2021 (2021-09-03) (Venice)  October 22, 2021 (2021-10-22) (United
States) | [...] The film is the third adaptation of Dune, following David Lynch's 1984 film "Dune (1984 film)") and
John Harrison "John Harrison (director)")'s 2000 television miniseries. After an unsuccessful attempt by Paramount 
Pictures to produce a new adaptation, Legendary Pictures acquired the Dune film and television rights in 2016, with
Villeneuve signing on as director in February 2017. Production contracts were secured only for the first film,     
relying on its success before a sequel would be produced. Principal photography took place from March to July 2019 
at locations including Budapest, Jordan, Norway, and Abu Dhabi. [...] Following the film's success, Warner Bros.   
and Legendary Pictures officially greenlit Dune: Part Two in October 2021. Villeneuve's main concern was to finish 
the production, which would benefit from all the work on the first part. Main characters reprise their role from   
the first film, with additional casting lasting from March to July 2022, and concluding by January 2023.           
Preliminary filming began in Italy by early July 2022, and concluded that December. The film was released on March 
1, 2024. Dune: Part Two's world premiere took place at Odeon Luxe Leicester Square in London on February 15.",     
"score": 0.9974491}, {"title": "Joshua Grode | Dune Wiki | Fandom", "url":                                         
"https://dune.fandom.com/wiki/Joshua_Grode", "content": "## General overview[]\n\nAs CEO, Grode oversees Legendary 
Pictures, which in recent years has released the blockbuster hits Godzilla vs. Kong and Pokémon Detective Pikachu, 
as well as Legendary Television (Netflix’s Lost in Space, Amazon’s Carnival Row), Legendary Comics and divisions   
covering digital media and VR. On April 2019, Grode stated that they plan to make a sequel to the first Dune,      
adding that "there's a logical place to stop the [first] movie before the book "Dune (novel)") is over".\n\n##     
External links[]\n\n Joshua Grode on the Internet Movie Database\n\n## References[] [...] ## References[]\n\n1. ↑  
Joshua Grode Takes Legendary CEO Post; How He And Mary Parent Intend To Write Wanda-Backed Company’s Next Chapter -
Deadline\n2. ↑ Legendary Entertainment CEO Joshua Grode on ‘Dune’ Success and Optimism for Future Theatrical       
Releases - Variety\n3. ↑ Legendary CEO Joshua Grode on Pitting ‘Pikachu’ Against Marvel, Warner Bros. Upheaval -   
Hollywood Reporter\n\nCategories\n\nCommunity content is available under CC-BY-SA unless otherwise noted.\n\nMore  
Fandoms \n\n Fantasy\n Sci-fi\n Dune [...] Dune Wiki\n\nDune Wiki is discussing some big overhaul projects to      
improve the site -- and we could use your help! Get involved here!\n\nREAD MORE\n\nDune Wiki\n\nSign In \n\nDon't  
have an account?\n\n Register  \n\n  Sign In\n\nSkip to content\n\nDune Wiki\n\n5,024\n\npages\n\nin: Crew\n\n#    
Joshua Grode\n\nSign in to edit \n\n History\n Purge\n Talk (0)\n\nJoshua Grode is a film producer and the current 
CEO of Legendary Entertainment as of 2017.\n\n## General

**输出讨论：**
正如预期的那样，基本智能体失败了。它的单次工具调用可能是搜索整个长查询。对于这种复杂的、联合查询，搜索结果通常很混乱，不会在一个地方包含所有必要的信息片段。

智能体的最终答案可能不完整、不正确，或者声明它找不到信息。它无法分解问题：
1. 找到制作《沙丘》的公司（传奇影业）。
2. 找到该公司的 CEO（Joshua Grode）。
3. 找到该公司最近的电影及其预算。

这一失败完美地说明了对更动态方法的需求。智能体需要一种方法来**对**它在一步中找到的信息做出**反应**，以指导下一步。

## 阶段 2：高级方法 - 实现 ReAct

现在，我们将构建真正的 ReAct 智能体。核心区别在于图的结构：我们将引入一个循环，允许智能体反复思考、行动和观察。

### 步骤 2.1：构建 ReAct 智能体图

**我们要做什么：**
我们将定义节点和创建 `思考 → 行动` 循环的关键路由器函数。关键架构变化是将 `tool_node` 的输出*返回*到 `agent_node` 的边，允许智能体查看结果并决定其下一步。

In [12]:
def react_agent_node(state: AgentState):
    console.print("--- REACT AGENT: Thinking... ---")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# The ToolNode is the same as before
react_tool_node = ToolNode([search_tool])

# The router is also the same logic
def react_router(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        console.print("--- ROUTER: Decision is to call a tool. ---")
        return "tools"
    console.print("--- ROUTER: Decision is to finish. ---")
    return "__end__"

# Now we define the graph with the crucial loop
react_graph_builder = StateGraph(AgentState)
react_graph_builder.add_node("agent", react_agent_node)
react_graph_builder.add_node("tools", react_tool_node)

react_graph_builder.set_entry_point("agent")
react_graph_builder.add_conditional_edges("agent", react_router, {"tools": "tools", "__end__": "__end__"})

# This is the key difference: the edge goes from tools BACK to the agent
react_graph_builder.add_edge("tools", "agent")

react_agent_app = react_graph_builder.compile()
print("ReAct agent compiled successfully with a reasoning loop.")

ReAct agent compiled successfully with a reasoning loop.


## 阶段 3：正面比较

现在我们将使用新的 ReAct 智能体运行相同的复杂查询，并观察其过程和最终输出的差异。

### 步骤 3.1：在多步问题上测试 ReAct 智能体

**我们要做什么：**
我们将使用相同的多步查询调用 ReAct 智能体，并流式传输输出以查看其迭代推理过程。

In [13]:
console.print(f"[bold green]Testing ReAct agent on the same multi-step query:[/bold green] '{multi_step_query}'\n")

final_react_output = None
for chunk in react_agent_app.stream({"messages": [("user", multi_step_query)]}, stream_mode="values"):
    final_react_output = chunk
    console.print(f"--- [bold purple]Current State[/bold purple] ---")
    chunk['messages'][-1].pretty_print()
    console.print("\n")

console.print("\n--- [bold green]Final Output from ReAct Agent[/bold green] ---")
console.print(Markdown(final_react_output['messages'][-1].content))

Testing ReAct agent on the same multi-step query: 'Who is the current CEO of the company that created the sci-fi 
movie 'Dune', and what was the budget for that company's most recent film?'

--- Current State ---

================================ Human Message =================================

Who is the current CEO of the company that created the sci-fi movie 'Dune', and what was the budget for that company's most recent film?


--- REACT AGENT: Thinking... ---

--- ROUTER: Decision is to call a tool. ---

...

--- ROUTER: Decision is to finish. ---

--- Current State ---

================================== Ai Message ==================================

The company behind *Dune* (2021) is **Legendary Pictures / Legendary Entertainment**, and its current CEO is **Josh (Joshua B.) Grode**. (For example, Variety explicitly refers to him as “Legendary CEO Josh Grode.”)  
Source: https://variety.com/2025/film/news/street-fighter-release-date-paramount-legendary-deal-1236508468/

As for that company’s **most recently released film with a widely published production budget**, *A Minecraft Movie* (2025) is listed at a **$150 million** budget.  
Source: https://en.wikipedia.org/wiki/A_Minecraft_Movie


--- Final Output from ReAct Agent ---

The company behind Dune (2021) is Legendary Pictures / Legendary Entertainment, and its current CEO is Josh (Joshua
B.) Grode. (For example, Variety explicitly refers to him as “Legendary CEO Josh Grode.”)                          
Source: https://variety.com/2025/film/news/street-fighter-release-date-paramount-legendary-deal-1236508468/        

As for that company’s most recently released film with a widely published production budget, A Minecraft Movie     
(2025) is listed at a $150 million budget.                                                                         
Source: https://en.wikipedia.org/wiki/A_Minecraft_Movie

**输出讨论：**
成功！执行追踪显示了一个完全不同且更加智能的过程。您可以看到智能体的逐步推理：
1. **想法 1：** 它首先推理需要识别《沙丘》的制作公司。
2. **行动 1：** 它使用类似"《沙丘》电影制作公司"的查询调用 `web_search` 工具。
3. **观察 1：** 它收到结果："传奇影业"。
4. **想法 2：** 现在，整合新信息，它推理需要传奇影业的 CEO。
5. **行动 2：** 它使用类似"传奇影业 CEO"的查询再次调用 `web_search`。
6. ...以此类推，直到它收集了所有必要的片段。
7. **综合：** 最后，它将所有收集的事实组装成完整且准确的答案。

这清楚地证明了 ReAct 模式对于任何非简单单步查找的任务的优越性。

## 阶段 4：定量评估

为了正式化比较，我们将使用 LLM 作为评判者来评分基本智能体和 ReAct 智能体的最终输出，评估它们完成任务的能力。

In [ ]:
class TaskEvaluation(BaseModel):
    """Schema for evaluating an agent's ability to complete a task."""
    task_completion_score: int = Field(description="Score 1-10 on whether the agent successfully completed all parts of the user's request.")
    reasoning_quality_score: int = Field(description="Score 1-10 on the logical flow and reasoning process demonstrated by the agent.")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(TaskEvaluation)

def evaluate_agent_output(query: str, agent_output: dict):
    trace = "\n".join([f"{m.type}: {m.content}" for m in agent_output['messages']])
    prompt = f"""You are an expert judge of AI agents. Evaluate the following agent's performance on the given task on a scale of 1-10. A score of 10 means the task was completed perfectly. A score of 1 means complete failure.
    
    **User's Task:**
    {query}
    
    **Full Agent Conversation Trace:**
    ```
    {trace}
    ```
    """
    return judge_llm.invoke(prompt)

console.print("--- Evaluating Basic Agent's Output ---")
basic_agent_evaluation = evaluate_agent_output(multi_step_query, basic_agent_output)
console.print(basic_agent_evaluation.model_dump())

console.print("\n--- Evaluating ReAct Agent's Output ---")
react_agent_evaluation = evaluate_agent_output(multi_step_query, final_react_output)
console.print(react_agent_evaluation.model_dump())

--- Evaluating Basic Agent's Output ---

{
    'task_completion_score': 2,
    'reasoning_quality_score': 4,
    'justification': 'The agent retrieved relevant sources identifying Legendary Pictures as the production company
behind Dune (2021) and Joshua Grode as CEO of Legendary Entertainment, and it also pulled a budget figure for a 
recent Legendary film (Godzilla x Kong: The New Empire: $135–150M). However, the agent never produced an actual 
final response to the user, never explicitly connected the company-to-CEO-to-most-recent-film chain in prose, and 
did not resolve ambiguity about what counts as the company’s “most recent film” (e.g., Dune: Part Two vs. Godzilla 
x Kong). Because there is no delivered answer, task completion is very poor despite decent evidence gathering.'
}

--- Evaluating ReAct Agent's Output ---

**输出讨论：**
来自 LLM 作为评判者的定量分数使差异变得清晰明了。
- **基本智能体**获得了非常低的 `task_completion_score`，因为它未能收集所有必需的信息。其 `reasoning_quality_score` 也很低，因为其过程有缺陷且不完整。
- 相比之下，**ReAct 智能体**获得了接近完美的分数。评判者认识到其迭代过程使其能够成功完成复杂任务的所有部分。

这种正面比较和评估提供了 ReAct 架构价值的明确证明。它是开启智能体解决需要动态适应的复杂、多跳问题能力的关键。

## 结论

在本笔记本中，我们不仅实现了 **ReAct** 架构，还展示了其相对于更基本的单次方法的明显优势。通过构建一个允许智能体循环通过推理和行动周期的工作流程，我们使其能够解决否则难以解决的复杂、多步问题。

观察行动结果并使用该信息指导下一步的能力是智能行为的基本组成部分。ReAct 模式提供了一种简单而深刻有效的方法来将这种能力构建到我们的 AI 智能体中，使其对于现实世界任务更强大、更自适应、更有用。